## Retrieving Sample Data

In [2]:
import pandas as pd
import requests

url = "https://data.cityofchicago.org/resource/4ijn-s7e5.json"

params = {
    "$limit" : 1000
}

response = requests.get(url, params = params, timeout = 30)

response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

df.head()

,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,...,results,violations,latitude,longitude,location,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_vrxf_vc4k,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu
0,2645374,SIP & SAVOR COFFEE HOUSE,SIP & SAVOR,3002330,Restaurant,Risk 1 (High),528 E 43RD ST,CHICAGO,IL,60653,...,Pass w/ Conditions,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",41.816815018289546,-87.61292308115752,"{'latitude': '41.816815018289546', 'longitude'...",1,4301,4,583,10
1,2645376,TEENY TINY UNIVERSITY,TEENY TINY UNIVERSITY,2517601,Daycare Above and Under 2 Years,Risk 1 (High),6111 S COTTAGE GROVE AVE,CHICAGO,IL,60637,...,Pass,NaN,41.78368442665007,-87.6058547469109,"{'latitude': '41.78368442665007', 'longitude':...",53,22260,9,470,4
2,2645426,MIKKEY'S RETRO GRILL,MIKKEY'S RETRO GRILL,2615910,Restaurant,Risk 1 (High),8132 S STONY ISLAND AVE,CHICAGO,IL,60617,...,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.74684499702,-87.58601627170704,"{'latitude': '41.74684499702', 'longitude': '-...",9,21202,41,133,35
3,2645389,"GRAHAM CRACKERS DAYCARE CENTER III, INC","GRAHAM CRACKERS DAYCARE CENTER III,INC.",2785825,Children's Services Facility,Risk 1 (High),401 E 111TH ST,CHICAGO,IL,60628,...,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.69262847127151,-87.6133140522167,"{'latitude': '41.69262847127151', 'longitude':...",30,21861,45,253,43
4,2645430,BENTO,BENTO SUSHI,3020653,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637,...,Pass,NaN,41.79126061783551,-87.60508491200589,"{'latitude': '41.79126061783551', 'longitude':...",32,22260,8,154,33


## Inspecting Dataset

In [3]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
df.info()

Shape: (1000, 22)

Column names:
['inspection_id', 'dba_name', 'aka_name', 'license_', 'facility_type', 'risk', 'address', 'city', 'state', 'zip', 'inspection_date', 'inspection_type', 'results', 'violations', 'latitude', 'longitude', 'location', ':@computed_region_awaf_s7ux', ':@computed_region_6mkv_f3dw', ':@computed_region_vrxf_vc4k', ':@computed_region_bdys_3d7i', ':@computed_region_43wa_7qmu']

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   inspection_id                1000 non-null   str   
 1   dba_name                     1000 non-null   str   
 2   aka_name                     998 non-null    str   
 3   license_                     1000 non-null   str   
 4   facility_type                977 non-null    str   
 5   risk                         1000 non-null   str   
 6   address                      

## Identifying Missing Values

In [8]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

missing.sort_values("Missing Count", ascending=False)

,Missing Count,Missing %
violations,340,34.0
facility_type,23,2.3
:@computed_region_43wa_7qmu,11,1.1
:@computed_region_bdys_3d7i,11,1.1
:@computed_region_vrxf_vc4k,11,1.1
:@computed_region_6mkv_f3dw,11,1.1
:@computed_region_awaf_s7ux,11,1.1
location,11,1.1
longitude,11,1.1
latitude,11,1.1


## Checking Duplicates

In [10]:
import json

df_for_duplicates = df.copy()
df_for_duplicates["location"] = df_for_duplicates["location"].map(
    lambda value: json.dumps(value, sort_keys=True)
    if isinstance(value, dict) else value
)

print("Fully duplicated rows:", df_for_duplicates.duplicated().sum())

if "inspection_id" in df.columns:
    print("Duplicated inspection IDs:",
          df["inspection_id"].duplicated().sum())

Fully duplicated rows: 0
Duplicated inspection IDs: 0


## Inspecting Categorical Values

In [11]:
for col in ["results", "risk", "facility_type"]:
    if col in df.columns:
        print(f"\n{col.upper()})")
        print(df[col].value_counts(dropna=False))


RESULTS)
results
Pass                  559
Fail                  222
Pass w/ Conditions     91
No Entry               45
Not Ready              42
Out of Business        41
Name: count, dtype: int64

RISK)
risk
Risk 1 (High)      759
Risk 2 (Medium)    144
Risk 3 (Low)        90
All                  7
Name: count, dtype: int64

FACILITY_TYPE)
facility_type
Restaurant                         579
School                             143
Grocery Store                      108
Children's Services Facility        48
Daycare Above and Under 2 Years     28
NaN                                 23
Bakery                              13
Mobile Food Preparer                10
Liquor                               6
Hospital                             6
CHARTER SCHOOL                       5
Daycare (2 - 6 Years)                5
Wholesale                            3
Long Term Care                       3
Golden Diner                         3
Catering                             2
Shared Kitchen  